In [ ]:
import yaml
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

from utils.features import engineer_features_fold
from utils.labels import generate_stress_targets_for_fold

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

BASE_DATA_PATH = "../data/processed/dataset_features_base.csv"
ES_PATH = "../data/processed/ES_D.csv"
CONFIG_PATH = "../config/model_config.yaml"
OUTPUT_DIR = Path("../artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Load config
# ------------------------------------------------------------------

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# ------------------------------------------------------------------
# Load data
# ------------------------------------------------------------------

base_df = pd.read_csv(BASE_DATA_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
es_df   = pd.read_csv(ES_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

print("Loaded base data:", base_df.shape)
print("Loaded ES data:", es_df.shape)
print("Horizons:", config["horizons"])
print("Models:", [m for m, v in config["models"].items() if v["enabled"]])

In [ ]:
import numpy as np
import yaml
import logging
import warnings
from pathlib import Path
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.exceptions import UndefinedMetricWarning

import xgboost as xgb

# =============================================================================
# WARNINGS
# =============================================================================

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# LOGGING SETUP
# =============================================================================

LOG_DIR = Path("../logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_DIR / "training.log", encoding="utf-8")
file_handler.setFormatter(formatter)
file_handler.setLevel(logging.DEBUG)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
stream_handler.setLevel(logging.INFO)

logger.handlers = []
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

# =============================================================================
# MODEL FACTORY
# =============================================================================

def build_model(name, params):
    if name == "logistic_regression":
        return LogisticRegression(**params)
    if name == "xgboost":
        return xgb.XGBClassifier(**params)
    if name == "catboost":
        return CatBoostClassifier(**params)
    raise ValueError(f"Unknown model: {name}")

# =============================================================================
# WALK-FORWARD SPLITS (PURGED EXPANDING)
# =============================================================================

years = base_df["date"].dt.year
unique_years = sorted(years.unique())

EMBARGO_DAYS = max(config["horizons"])
splits = []

for i in range(2, len(unique_years)):
    train_years = unique_years[:i]
    test_year = unique_years[i]

    train_idx = base_df[years.isin(train_years)].index
    test_idx = base_df[years == test_year].index

    if len(train_idx) > 0:
        embargo_start = test_idx.min() - EMBARGO_DAYS
        train_idx = train_idx[train_idx < embargo_start]

    splits.append((train_idx, test_idx, test_year))

logger.info(f"Prepared {len(splits)} walk-forward splits")

# =============================================================================
# GRID SEARCH
# =============================================================================

best_params = {}

for horizon in config["horizons"]:
    logger.info(f"START Horizon={horizon}")
    best_params[horizon] = {}

    for model_name, model_cfg in config["models"].items():
        if not model_cfg["enabled"]:
            continue
        
        if model_name == "garch":
            continue

        logger.info(f"Model={model_name}")

        grid = ParameterGrid(model_cfg["param_grid"])
        model_scores = []

        for params in grid:
            fold_scores = []
            skipped_years = set()

            for train_idx, test_idx, test_year in splits:
                df_train = base_df.loc[train_idx]
                df_test  = base_df.loc[test_idx]
                
                # -------- Feature engineering --------
                X_train, X_test = engineer_features_fold(df_train, df_test)
                if X_train.empty or X_test.empty:
                    continue

                # DROP TOP 3 FEATURES
                drop_cols = [
                    "es_vol_14d", "es_vol_7d",
                    "es_ret_z_20", "es_ret", "es_range",
                    "vix_chg",
                    "dxy_ret_z",
                    "US10Y_chg_5d", "TED_SPREAD_chg_5d",
                    "pct_negative",
                    # base sentiment
                    'sentiment_mean', 'sentiment_median', 'sentiment_std', 'sentiment_min',
                    'pct_negative', 'article_count', 'stress_article_count',
                    'calm_article_count', 'stress_ratio',
                    # engineered sentiment
                    'sentiment_change',
                    'sentiment_mean_roll_5', 'sentiment_mean_roll_10',
                    'sentiment_std_roll_5', 'sentiment_std_roll_10',
                    'stress_ratio_roll_5', 'stress_ratio_roll_10',
                    'sentiment_zscore_10',
                    'stress_ratio_change',
                    # lags
                    'sentiment_mean_lag1', 'sentiment_mean_lag2',
                    'sentiment_std_lag1', 'sentiment_std_lag2',
                    'pct_negative_lag1', 'pct_negative_lag2',
                    'sentiment_change_lag1', 'sentiment_change_lag2',
                    'sentiment_zscore_10_lag1', 'sentiment_zscore_10_lag2',
                    'stress_ratio_lag1', 'stress_ratio_lag2',
                    'stress_ratio_change_lag1', 'stress_ratio_change_lag2',
                ]
                X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
                X_test = X_test.drop(columns=[c for c in drop_cols if c in X_test.columns])

                # -------- Labels --------
                y = generate_stress_targets_for_fold(
                    es_df=es_df,
                    train_end=df_train["date"].max(),
                    horizon=horizon,
                )

                y_train = y.loc[X_train["date"]].dropna()
                y_test  = y.loc[X_test["date"]].dropna()

                X_tr = X_train[X_train["date"].isin(y_train.index)].drop(columns=["date"])
                X_te = X_test[X_test["date"].isin(y_test.index)].drop(columns=["date"])

                if y_train.nunique() < 2 or y_train.sum() < 5 or len(y_test) == 0:
                    if test_year not in skipped_years:
                        logger.debug(
                            f"SKIP | H{horizon} | {model_name} | "
                            f"FoldYear={test_year} | Positives={y_train.sum()}"
                        )
                        skipped_years.add(test_year)
                    continue
                
                if y_test.sum() == 0:
                    logger.debug(
                        f"SKIP-SCORE | H{horizon} | {model_name} | "
                        f"FoldYear={test_year} | No positives in y_test"
                    )
                    continue

                model_params = params.copy()

                if model_name == "xgboost" and model_params.get("scale_pos_weight") == "auto":
                    weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
                    model_params["scale_pos_weight"] = min(weight, 50)

                if model_name == "catboost" and model_params.get("auto_class_weights") == "Balanced":
                    pass

                model = build_model(model_name, model_params)
                model.fit(X_tr, y_train)

                proba = model.predict_proba(X_te)[:, 1]
                score = average_precision_score(y_test, proba)
                fold_scores.append(score)

                logger.debug(
                    f"H{horizon} | {model_name} | "
                    f"FoldYear={test_year} | PR-AUC={score:.4f}"
                )

            if fold_scores:
                model_scores.append(
                    {
                        "mean_pr_auc": float(np.mean(fold_scores)),
                        "std_pr_auc": float(np.std(fold_scores)),
                        "params": model_params,
                    }
                )

        logger.info(
            f"COMPLETED | H{horizon} | {model_name} | "
            f"Evaluated {len(model_scores)} valid configs"
        )

        if not model_scores:
            logger.info(f"No valid scores for H{horizon} | {model_name}")
            continue

        best = max(model_scores, key=lambda x: x["mean_pr_auc"])
        best_params[horizon][model_name] = best

        logger.info(
            f"BEST | H{horizon} | {model_name} | "
            f"PR-AUC={best['mean_pr_auc']:.4f} ± {best['std_pr_auc']:.4f}"
        )

# =============================================================================
# SAVE BEST PARAMETERS
# =============================================================================

BEST_PARAM_PATH = Path("../artifacts/ablation/ablation_best_model_params.yaml")
BEST_PARAM_PATH.parent.mkdir(parents=True, exist_ok=True)

# Convert numpy types to Python types
def clean_for_yaml(obj):
    if isinstance(obj, dict):
        return {k: clean_for_yaml(v) for k, v in obj.items()}
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

best_params = clean_for_yaml(best_params)

with open(BEST_PARAM_PATH, "w") as f:
    yaml.safe_dump(best_params, f)

logger.info(f"Saved best hyperparameters -> {BEST_PARAM_PATH}")

In [ ]:
# =============================================================================
# CELL 3: TRAIN FINAL MODELS + OOF PREDICTIONS + THRESHOLD TUNING
# =============================================================================

import pickle
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
from sklearn.metrics import f1_score

MODEL_DIR = Path("../data/ablation/models")
OOF_DIR = Path("../data/ablation/oof")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)

DROP_COLS = [
    'es_vol_14d', 'es_vol_7d',
    "es_ret_z_20", "es_ret", "es_range",
    "vix_chg",
    "dxy_ret_z",
    "US10Y_chg_5d", "TED_SPREAD_chg_5d",
    "pct_negative",
    # base sentiment
    'sentiment_mean', 'sentiment_median', 'sentiment_std', 'sentiment_min',
    'pct_negative', 'article_count', 'stress_article_count',
    'calm_article_count', 'stress_ratio',
    # engineered sentiment
    'sentiment_change',
    'sentiment_mean_roll_5', 'sentiment_mean_roll_10',
    'sentiment_std_roll_5', 'sentiment_std_roll_10',
    'stress_ratio_roll_5', 'stress_ratio_roll_10',
    'sentiment_zscore_10',
    'stress_ratio_change',
    # lags
    'sentiment_mean_lag1', 'sentiment_mean_lag2',
    'sentiment_std_lag1', 'sentiment_std_lag2',
    'pct_negative_lag1', 'pct_negative_lag2',
    'sentiment_change_lag1', 'sentiment_change_lag2',
    'sentiment_zscore_10_lag1', 'sentiment_zscore_10_lag2',
    'stress_ratio_lag1', 'stress_ratio_lag2',
    'stress_ratio_change_lag1', 'stress_ratio_change_lag2',
]

# Load best params
with open("../artifacts/ablation/ablation_best_model_params.yaml", "r") as f:
    best_params = yaml.safe_load(f)

# Full training data 2019-2024
train_data = base_df[base_df["date"].dt.year <= 2024]

logger.info(f"Training final models on {len(train_data)} rows (2019-2024)")

for horizon in config["horizons"]:
    logger.info(f"Horizon={horizon}")

    for model_name in ["logistic_regression", "xgboost", "catboost"]:
        if model_name not in best_params.get(horizon, {}):
            continue

        logger.info(f"  Model={model_name}")

        # Get best params
        params = best_params[horizon][model_name]["params"]

        # Engineer features on full train set
        X_full, _ = engineer_features_fold(train_data, train_data.iloc[:0])
        X_full = X_full.drop(columns=[c for c in DROP_COLS if c in X_full.columns])

        # Generate labels
        y = generate_stress_targets_for_fold(
            es_df=es_df,
            train_end=train_data["date"].max(),
            horizon=horizon,
        )

        y_full = y.loc[X_full["date"]].dropna()
        X_tr = X_full[X_full["date"].isin(y_full.index)].drop(columns=["date"])

        # Auto scale_pos_weight for XGBoost
        if model_name == "xgboost" and params.get("scale_pos_weight") == "auto":
            weight = (y_full == 0).sum() / max((y_full == 1).sum(), 1)
            params["scale_pos_weight"] = min(weight, 50)

        # Train
        model = build_model(model_name, params)
        model.fit(X_tr, y_full)

        # Save model
        model_path = MODEL_DIR / f"{model_name}_h{horizon}.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(model, f)

        logger.info(f"    Saved -> {model_path}")

        # Generate OOF predictions
        oof_preds = []
        oof_dates = []
        oof_actuals = []

        for train_idx, test_idx, test_year in splits:
            df_train = base_df.loc[train_idx]
            df_test = base_df.loc[test_idx]

            X_train, X_test = engineer_features_fold(df_train, df_test)
            if X_train.empty or X_test.empty:
                continue

            X_train = X_train.drop(columns=[c for c in DROP_COLS if c in X_train.columns])
            X_test = X_test.drop(columns=[c for c in DROP_COLS if c in X_test.columns])

            y = generate_stress_targets_for_fold(
                es_df=es_df,
                train_end=df_train["date"].max(),
                horizon=horizon,
            )

            y_train = y.loc[X_train["date"]].dropna()
            y_test = y.loc[X_test["date"]].dropna()

            X_tr = X_train[X_train["date"].isin(y_train.index)].drop(columns=["date"])
            X_te = X_test[X_test["date"].isin(y_test.index)].drop(columns=["date"])

            if y_train.nunique() < 2 or len(y_test) == 0:
                continue

            # Train fold model
            fold_params = params.copy()
            if model_name == "xgboost" and fold_params.get("scale_pos_weight") == "auto":
                weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
                fold_params["scale_pos_weight"] = min(weight, 50)

            fold_model = build_model(model_name, fold_params)
            fold_model.fit(X_tr, y_train)

            proba = fold_model.predict_proba(X_te)[:, 1]

            oof_preds.extend(proba)
            oof_dates.extend(X_test[X_test["date"].isin(y_test.index)]["date"])
            oof_actuals.extend(y_test)

        # Save OOF
        oof_df = pd.DataFrame({
            "date": oof_dates,
            "actual": oof_actuals,
            "predicted": oof_preds,
        })

        oof_path = OOF_DIR / f"{model_name}_h{horizon}_oof.csv"
        oof_df.to_csv(oof_path, index=False)
        logger.info(f"    OOF saved -> {oof_path} ({len(oof_df)} predictions)")

logger.info("Final training complete")

# =============================================================================
# THRESHOLD TUNING FROM OOF
# =============================================================================

optimal_thresholds = {}

for horizon in config["horizons"]:
    optimal_thresholds[horizon] = {}

    for model_name in ["logistic_regression", "xgboost", "catboost"]:
        oof_path = OOF_DIR / f"{model_name}_h{horizon}_oof.csv"
        if not oof_path.exists():
            continue

        oof = pd.read_csv(oof_path)
        y_true = oof["actual"].values
        y_prob = oof["predicted"].values

        best_t, best_f1 = 0.5, -1.0
        for t in np.linspace(0.01, 0.99, 99):
            score = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_t = float(np.round(t, 2))

        optimal_thresholds[horizon][model_name] = best_t
        logger.info(
            f"H{horizon} | {model_name} | Best threshold={best_t:.2f} | OOF F1={best_f1:.3f}"
        )

print(optimal_thresholds)

In [ ]:
# =============================================================================
# CELL 4: TEST ON 2025 (OUT-OF-SAMPLE)
# =============================================================================

import pickle
import os
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, recall_score, f1_score

DROP_COLS = [
    "es_vol_14d", "es_vol_7d",
    "es_ret_z_20", "es_ret", "es_range",
    "vix_chg",
    "dxy_ret_z",
    "US10Y_chg_5d", "TED_SPREAD_chg_5d",
    "pct_negative",
    # base sentiment
    'sentiment_mean', 'sentiment_median', 'sentiment_std', 'sentiment_min',
    'pct_negative', 'article_count', 'stress_article_count',
    'calm_article_count', 'stress_ratio',
    # engineered sentiment
    'sentiment_change',
    'sentiment_mean_roll_5', 'sentiment_mean_roll_10',
    'sentiment_std_roll_5', 'sentiment_std_roll_10',
    'stress_ratio_roll_5', 'stress_ratio_roll_10',
    'sentiment_zscore_10',
    'stress_ratio_change',
    # lags
    'sentiment_mean_lag1', 'sentiment_mean_lag2',
    'sentiment_std_lag1', 'sentiment_std_lag2',
    'pct_negative_lag1', 'pct_negative_lag2',
    'sentiment_change_lag1', 'sentiment_change_lag2',
    'sentiment_zscore_10_lag1', 'sentiment_zscore_10_lag2',
    'stress_ratio_lag1', 'stress_ratio_lag2',
    'stress_ratio_change_lag1', 'stress_ratio_change_lag2',
]

OOS_DIR = "../artifacts/ablation/ml_models_oos"
os.makedirs(OOS_DIR, exist_ok=True)

test_data = base_df[base_df["date"].dt.year == 2025]
logger.info(f"Testing on {len(test_data)} rows (2025)")

results = []

for horizon in config["horizons"]:
    logger.info(f"Horizon={horizon}")

    for model_name in ["logistic_regression", "xgboost", "catboost"]:

        model_path = MODEL_DIR / f"{model_name}_h{horizon}.pkl"
        if not model_path.exists():
            continue

        with open(model_path, "rb") as f:
            model = pickle.load(f)

        # Recreate test features exactly as training pipeline
        _, X_test = engineer_features_fold(
            base_df[base_df["date"].dt.year <= 2024],
            test_data
        )

        X_test = X_test.drop(
            columns=[c for c in DROP_COLS if c in X_test.columns]
        )

        if X_test.empty:
            continue

        y = generate_stress_targets_for_fold(
            es_df=es_df,
            train_end=base_df[base_df["date"].dt.year <= 2024]["date"].max(),
            horizon=horizon,
        )

        y_test = y.loc[X_test["date"]].dropna()
        X_te = X_test[X_test["date"].isin(y_test.index)].copy()

        if len(y_test) == 0 or y_test.sum() == 0:
            continue

        X_te_model = X_te.drop(columns=["date"])

        proba = model.predict_proba(X_te_model)[:, 1]

        threshold = optimal_thresholds[horizon][model_name]
        y_pred = (proba >= threshold).astype(int)

        # =========================
        # SAVE DAILY PREDICTIONS
        # =========================
        pred_df = pd.DataFrame({
            "date": X_te["date"].values,
            "y_true": y_test.values,
            "proba": proba,
            "pred": y_pred
        })

        pred_df.to_csv(
            f"{OOS_DIR}/{model_name}_ablation_oos_predictions_h{horizon}.csv",
            index=False
        )

        # =========================
        # METRICS
        # =========================
        pr_auc = average_precision_score(y_test, proba)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)

        top_5_idx = np.argsort(proba)[-int(0.05 * len(proba)):]
        top_10_idx = np.argsort(proba)[-int(0.10 * len(proba)):]

        prec_5 = y_test.iloc[top_5_idx].mean()
        prec_10 = y_test.iloc[top_10_idx].mean()

        results.append({
            "horizon": horizon,
            "model": model_name,
            "threshold": threshold,
            "pr_auc": pr_auc,
            "f1": f1,
            "recall": rec,
            "prec@5%": prec_5,
            "prec@10%": prec_10,
        })

        logger.info(
            f"  {model_name}: t={threshold} | PR-AUC={pr_auc:.3f} | "
            f"F1={f1:.3f} | Recall={rec:.3f} | "
            f"Prec@5%={prec_5:.3f} | Prec@10%={prec_10:.3f}"
        )

results_df = pd.DataFrame(results)
results_df.to_csv("../artifacts/ablation/ml_models_oos/oos_results_2025.csv", index=False)

logger.info(f"Saved -> ../artifacts/ablation/ml_models_oos/oos_results_2025.csv")
print("\n" + "=" * 60)
print(results_df.to_string(index=False))